# Coordinated Multi-Agent Recommendation System

This notebook integrates:

1. Query Agent
2. Retrieval Agent
3. Review Agent
4. Ranking Agent
5. Verifier Agent

Pipeline:

User Query → Query Agent → Retrieval Agent → Review Agent →
Ranking Agent → Verifier Agent → Final Recommendations

In [1]:
import os
import sys
import time
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv

from langchain_openai import (
    ChatOpenAI,
    OpenAIEmbeddings
)

from langchain_community.vectorstores import FAISS

C:\Users\srush\AppData\Local\Temp\ipykernel_21376\1471384404.py:14: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
load_dotenv()

True

In [3]:
PROJECT_ROOT = Path("..").resolve()

In [4]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\srush\Desktop\Multi agent coordination


In [5]:
import Agents.verifier_agent as verifier_module

print(verifier_module.__file__)
print(
    [
        name
        for name in dir(verifier_module)
        if "Verifier" in name
    ]
)

C:\Users\srush\Desktop\Multi agent coordination\Agents\verifier_agent.py
['VerifierAgent', 'VerifierOutput']


In [6]:
from Agents.query_agent import QueryAgent
from Agents.review_agent import ReviewAgent
from Agents.ranking_agent import RankingAgent
from Agents.verifier_agent import VerifierAgent
from Agents.retrieval_agent import (
    RetrievalAgent,
    build_search_query,
    filter_products
)

print("All agent classes imported successfully.")

All agent classes imported successfully.


In [7]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

print("LLM initialised successfully.")

LLM initialised successfully.


In [8]:
reviews_path = (
    PROJECT_ROOT
    / "Data"
    / "Cleaned"
    / "reviews_sample.parquet"
)

if not reviews_path.exists():
    raise FileNotFoundError(
        f"Reviews file not found at: {reviews_path}"
    )

reviews_df = pd.read_parquet(reviews_path)

print("Reviews shape:", reviews_df.shape)
print("Review columns:", reviews_df.columns.tolist())

Reviews shape: (26031, 15)
Review columns: ['rating', 'product_title', 'review_title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'review_date', 'price', 'average_rating', 'rating_number']


In [9]:
required_review_columns = {
    "parent_asin",
    "rating",
    "text"
}

missing_columns = (
    required_review_columns
    - set(reviews_df.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing review columns: {missing_columns}"
    )

print("Required review columns are available.")

Required review columns are available.


In [10]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print("Embeddings initialised.")

Embeddings initialised.


In [11]:
faiss_path = (
    PROJECT_ROOT
    / "Data"
    / "Cleaned"
    / "faiss_index"
)

if not faiss_path.exists():
    raise FileNotFoundError(
        f"FAISS index not found at: {faiss_path}"
    )

vector_db = FAISS.load_local(
    folder_path=str(faiss_path),
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)

print("FAISS vector database loaded.")

FAISS vector database loaded.


In [12]:
query_agent = QueryAgent()

retrieval_agent = RetrievalAgent(
    vector_db=vector_db
)

review_agent = ReviewAgent(
    llm=llm,
    reviews_df=reviews_df
)

ranking_agent = RankingAgent(
    llm=llm,
    explanation_limit=2,
    use_llm_explanations=True
)

verifier_agent = VerifierAgent(
    llm=llm,
    use_llm_summary=False
)

print("All five agents initialised successfully.")

All five agents initialised successfully.


In [13]:
#testing query agent: 

user_query = (
    "Recommend an Apple headphone with a good sound quality "
    "and long battery life."
)

structured_query = query_agent.parse(
    user_query
)

print(structured_query)

product_type='headset' brand='Apple' budget=None features=['good sound quality', 'long battery life']


In [14]:
#testing retrieval agent

retrieved_products = retrieval_agent.retrieve(
    query=structured_query,
    k=50
)

print(
    "Number of retrieved products:",
    len(retrieved_products)
)


Number of retrieved products: 3


In [15]:
#display results: 
for index, (document, score) in enumerate(
    retrieved_products,
    start=1
):
    print("=" * 80)
    print("Result:", index)
    print("Title:", document.metadata.get("title"))
    print(
        "Parent ASIN:",
        document.metadata.get("parent_asin")
    )
    print("Brand:", document.metadata.get("brand"))
    print("Price:", document.metadata.get("price"))
    print(
        "Average rating:",
        document.metadata.get("average_rating")
    )
    print("Retrieval score:", round(float(score), 4))

Result: 1
Title: vodbov Bluetooth Headset, Wireless earpiece for Business/Sport/Driver Noise Cancelling Mic Black Bluetooth Headphones Compatible with Apple Android Cell Phones,MacBook and Compatible with Alexa
Parent ASIN: B07GP25L9F
Brand: apple
Price: nan
Average rating: 3.9
Retrieval score: 0.4752
Result: 2
Title: Travel 3 in 1 Magsafe Wireless Charger, Foldable Wireless Charging Station for Apple, Wireless Charging Pad Compatible with iPhone 14 13 12 11/Pro/XS/XR,AirPods 3/2/Pro, iWatch 7/6/5/4/3/2
Parent ASIN: B0BLRVWPF2
Brand: apple
Price: 25.99
Average rating: 4.6
Retrieval score: 0.4546
Result: 3
Title: Foldable Wireless Charging Station for Multiple Devices Apple, 3 in 1 Wireless Charger for iPhone 13/12/11 Pro Max/X/Xs Max/8/8 Plus,for iWatch 7/6/5/4/3/2/se, AirPods 3/2/pro (Switchable Light)
Parent ASIN: B0BN5838R7
Brand: apple
Price: 21.99
Average rating: 4.1
Retrieval score: 0.4535


In [16]:
def enrich_products_with_reviews(
    retrieved_products: list,
    review_agent: ReviewAgent,
    max_reviews: int = 10
) -> list[dict]:

    enriched_products = []

    for document, retrieval_score in retrieved_products:

        metadata = document.metadata.copy()

        parent_asin = metadata.get("parent_asin")

        if not parent_asin:
            continue

        review_result = (
            review_agent.summarize_reviews(
                parent_asin=parent_asin,
                max_reviews=max_reviews
            )
        )

        enriched_product = {
            # Product identity
            "parent_asin": str(parent_asin),
            "title": str(
                metadata.get(
                    "title",
                    review_result.get(
                        "title",
                        "Unknown Product"
                    )
                )
            ),

            # Product metadata
            "brand": metadata.get("brand"),
            "product_type": metadata.get(
                "product_type"
            ),
            "price": metadata.get("price"),
            "categories": metadata.get(
                "categories"
            ),
            "product_text": document.page_content,

            # Retrieval evidence
            "retrieval_score": float(
                retrieval_score
            ),
            "similarity": metadata.get(
                "similarity",
                0
            ),

            # Product-level rating information
            "average_rating": float(
                metadata.get(
                    "average_rating",
                    review_result.get(
                        "average_rating",
                        0
                    )
                )
                or 0
            ),
            "rating_number": float(
                metadata.get(
                    "rating_number",
                    0
                )
                or 0
            ),

            # Review Agent output
            "review_count": review_result.get(
                "review_count",
                0
            ),
            "pros": review_result.get(
                "pros",
                []
            ),
            "cons": review_result.get(
                "cons",
                []
            ),
            "overall_sentiment": (
                review_result.get(
                    "overall_sentiment",
                    "Unknown"
                )
            ),
            "recommended_for": (
                review_result.get(
                    "recommended_for",
                    ""
                )
            ),
            "avoid_if": review_result.get(
                "avoid_if",
                ""
            ),
            "review_summary": review_result.get(
                "summary",
                ""
            ),
            "sample_reviews": review_result.get(
                "sample_reviews",
                ""
            )
        }

        enriched_products.append(
            enriched_product
        )

    return enriched_products

In [17]:
enriched_products = enrich_products_with_reviews(
    retrieved_products=retrieved_products,
    review_agent=review_agent,
    max_reviews=5
)

print(
    "Number of enriched products:",
    len(enriched_products)
)

Number of enriched products: 3


In [18]:
for index, product in enumerate(
    enriched_products,
    start=1
):
    print("=" * 80)
    print("Product:", index)
    print("Title:", product["title"])
    print(
        "Parent ASIN:",
        product["parent_asin"]
    )
    print(
        "Retrieval score:",
        round(product["retrieval_score"], 4)
    )
    print(
        "Average rating:",
        product["average_rating"]
    )
    print(
        "Review count:",
        product["review_count"]
    )
    print(
        "Sentiment:",
        product["overall_sentiment"]
    )
    print("Pros:", product["pros"])
    print("Cons:", product["cons"])
    print(
        "Review summary:",
        product["review_summary"]
    )

Product: 1
Title: vodbov Bluetooth Headset, Wireless earpiece for Business/Sport/Driver Noise Cancelling Mic Black Bluetooth Headphones Compatible with Apple Android Cell Phones,MacBook and Compatible with Alexa
Parent ASIN: B07GP25L9F
Retrieval score: 0.4752
Average rating: 3.9
Review count: 2
Sentiment: Mixed
Pros: ['Great backup option for other Bluetooth devices']
Cons: ['Poor sound quality with incorrect spoken English', 'Limited range', 'Overall cheap build quality']
Review summary: The vodbov Bluetooth Headset serves as a budget-friendly backup option but suffers from poor sound quality and limited range, making it less suitable for serious use.
Product: 2
Title: Travel 3 in 1 Magsafe Wireless Charger, Foldable Wireless Charging Station for Apple, Wireless Charging Pad Compatible with iPhone 14 13 12 11/Pro/XS/XR,AirPods 3/2/Pro, iWatch 7/6/5/4/3/2
Parent ASIN: B0BLRVWPF2
Retrieval score: 0.4546
Average rating: 4.6
Review count: 5
Sentiment: Positive
Pros: ['Compact and portable

In [19]:
#running ranking agent

ranking_output = ranking_agent.rank_products(
    query=structured_query,
    products=enriched_products
)

print(ranking_output)

products=[RankedProduct(rank=1, parent_asin='B0BLRVWPF2', title='Travel 3 in 1 Magsafe Wireless Charger, Foldable Wireless Charging Station for Apple, Wireless Charging Pad Compatible with iPhone 14 13 12 11/Pro/XS/XR,AirPods 3/2/Pro, iWatch 7/6/5/4/3/2', average_rating=4.6, retrieval_score=0.45460320505695495, final_score=0.46124, reason='The Travel 3 in 1 MagSafe Wireless Charger is ranked first due to its compact design and ability to charge multiple devices simultaneously, which supports the requested feature of good sound quality indirectly by enhancing user experience with Apple devices. However, long battery life could not be verified from the available evidence.'), RankedProduct(rank=2, parent_asin='B0BN5838R7', title='Foldable Wireless Charging Station for Multiple Devices Apple, 3 in 1 Wireless Charger for iPhone 13/12/11 Pro Max/X/Xs Max/8/8 Plus,for iWatch 7/6/5/4/3/2/se, AirPods 3/2/pro (Switchable Light)', average_rating=4.1, retrieval_score=0.45353601702949853, final_sco

In [20]:
for product in ranking_output.products:

    print("=" * 80)
    print("Rank:", product.rank)
    print("Title:", product.title)
    print(
        "Parent ASIN:",
        product.parent_asin
    )
    print(
        "Average rating:",
        product.average_rating
    )
    print(
        "Retrieval score:",
        round(product.retrieval_score, 4)
    )
    print(
        "Final score:",
        round(product.final_score, 4)
    )
    print("Reason:", product.reason)

Rank: 1
Title: Travel 3 in 1 Magsafe Wireless Charger, Foldable Wireless Charging Station for Apple, Wireless Charging Pad Compatible with iPhone 14 13 12 11/Pro/XS/XR,AirPods 3/2/Pro, iWatch 7/6/5/4/3/2
Parent ASIN: B0BLRVWPF2
Average rating: 4.6
Retrieval score: 0.4546
Final score: 0.4612
Reason: The Travel 3 in 1 MagSafe Wireless Charger is ranked first due to its compact design and ability to charge multiple devices simultaneously, which supports the requested feature of good sound quality indirectly by enhancing user experience with Apple devices. However, long battery life could not be verified from the available evidence.
Rank: 2
Title: Foldable Wireless Charging Station for Multiple Devices Apple, 3 in 1 Wireless Charger for iPhone 13/12/11 Pro Max/X/Xs Max/8/8 Plus,for iWatch 7/6/5/4/3/2/se, AirPods 3/2/pro (Switchable Light)
Parent ASIN: B0BN5838R7
Average rating: 4.1
Retrieval score: 0.4535
Final score: 0.4135
Reason: The Foldable Wireless Charging Station is ranked second a

In [21]:
#running verifier agent:

verification_output = verifier_agent.verify(
    query=structured_query,
    ranking_output=ranking_output,
    original_products=enriched_products
)

print(verification_output)

overall_status='failed' confidence='low' verified_products=[ProductVerification(rank=1, parent_asin='B0BLRVWPF2', title='Travel 3 in 1 Magsafe Wireless Charger, Foldable Wireless Charging Station for Apple, Wireless Charging Pad Compatible with iPhone 14 13 12 11/Pro/XS/XR,AirPods 3/2/Pro, iWatch 7/6/5/4/3/2', status='failed', brand_match=True, product_type_match=False, feature_status='no_match', feature_match_ratio=0.0, matched_features=[], missing_features=['good sound quality', 'long battery life'], budget_verifiable=False, budget_match=None, evidence_strength='moderate', warnings=['The product may not match the requested product type or may be an accessory.', 'None of the requested features could be verified from the available evidence.', 'Missing feature evidence: good sound quality, long battery life'], verification_reason='Brand match: True. Product type match: False. Feature status: no_match. Feature match ratio: 0.00. Evidence strength: moderate. Price available: True. Budget 

In [22]:
print(
    "Overall status:",
    verification_output.overall_status
)

print(
    "Confidence:",
    verification_output.confidence
)

print(
    "Recommended product:",
    verification_output.recommended_product_title
)

print(
    "Recommended ASIN:",
    verification_output.recommended_product_asin
)

print("\nSummary:")
print(verification_output.summary)

Overall status: failed
Confidence: low
Recommended product: None
Recommended ASIN: None

Summary:
Verification finished with status 'failed' and low confidence. All 3 products failed one or more critical checks, so no product could be recommended.


In [23]:
for product in verification_output.verified_products:

    print("=" * 80)
    print("Rank:", product.rank)
    print("Title:", product.title)
    print("Status:", product.status)
    print("Brand match:", product.brand_match)
    print(
        "Product type match:",
        product.product_type_match
    )
    print(
        "Feature status:",
        product.feature_status
    )
    print(
        "Feature match ratio:",
        product.feature_match_ratio
    )
    print(
        "Evidence strength:",
        product.evidence_strength
    )
    print("Warnings:", product.warnings)

Rank: 1
Title: Travel 3 in 1 Magsafe Wireless Charger, Foldable Wireless Charging Station for Apple, Wireless Charging Pad Compatible with iPhone 14 13 12 11/Pro/XS/XR,AirPods 3/2/Pro, iWatch 7/6/5/4/3/2
Status: failed
Brand match: True
Product type match: False
Feature status: no_match
Feature match ratio: 0.0
Evidence strength: moderate
Warnings: ['The product may not match the requested product type or may be an accessory.', 'None of the requested features could be verified from the available evidence.', 'Missing feature evidence: good sound quality, long battery life']
Rank: 2
Title: Foldable Wireless Charging Station for Multiple Devices Apple, 3 in 1 Wireless Charger for iPhone 13/12/11 Pro Max/X/Xs Max/8/8 Plus,for iWatch 7/6/5/4/3/2/se, AirPods 3/2/pro (Switchable Light)
Status: failed
Brand match: True
Product type match: False
Feature status: no_match
Feature match ratio: 0.0
Evidence strength: moderate
Warnings: ['The product may not match the requested product type or may b

In [24]:
#making coordinated pipeline:

def run_coordinated_system(
    user_query: str,
    retrieval_k: int = 15,
    max_reviews: int = 3,
    verbose: bool = True
) -> dict:

    if not user_query or not user_query.strip():
        raise ValueError(
            "User query cannot be empty."
        )

    pipeline_start = time.perf_counter()

    # ============================================
    # 1. Query Agent
    # ============================================

    query_start = time.perf_counter()

    structured_query = query_agent.parse(
        user_query
    )

    query_latency = (
        time.perf_counter()
        - query_start
    )

    if verbose:
        print("=" * 100)
        print("1. QUERY AGENT")
        print("=" * 100)
        print(structured_query)
        print()

    # ============================================
    # 2. Retrieval Agent
    # ============================================

    retrieval_start = time.perf_counter()

    retrieved_products = (
        retrieval_agent.retrieve(
            query=structured_query,
            k=retrieval_k
        )
    )

    retrieval_latency = (
        time.perf_counter()
        - retrieval_start
    )

    if verbose:
        print("=" * 100)
        print("2. RETRIEVAL AGENT")
        print("=" * 100)
        print(
            "Retrieved products:",
            len(retrieved_products)
        )

        for document, score in retrieved_products:
            print(
                "-",
                document.metadata.get("title"),
                "| Score:",
                round(float(score), 4)
            )

        print()

    # ============================================
    # 3. Review Agent
    # ============================================

    review_start = time.perf_counter()

    enriched_products = (
        enrich_products_with_reviews(
            retrieved_products=(
                retrieved_products
            ),
            review_agent=review_agent,
            max_reviews=max_reviews
        )
    )

    review_latency = (
        time.perf_counter()
        - review_start
    )

    if verbose:
        print("=" * 100)
        print("3. REVIEW AGENT")
        print("=" * 100)
        print(
            "Products enriched:",
            len(enriched_products)
        )

        for product in enriched_products:
            print(
                "-",
                product["title"],
                "| Sentiment:",
                product["overall_sentiment"]
            )

        print()

    # Stop safely if retrieval produced no valid products
    if not enriched_products:

        total_latency = (
            time.perf_counter()
            - pipeline_start
        )

        return {
            "system": "coordinated_multi_agent",
            "user_query": user_query,
            "structured_query": structured_query,
            "retrieved_products": [],
            "enriched_products": [],
            "ranking_output": None,
            "verification_output": None,
            "latency": {
                "query_agent": round(
                    query_latency,
                    4
                ),
                "retrieval_agent": round(
                    retrieval_latency,
                    4
                ),
                "review_agent": round(
                    review_latency,
                    4
                ),
                "ranking_agent": 0,
                "verifier_agent": 0,
                "total": round(
                    total_latency,
                    4
                )
            },
            "error": (
                "No valid products were available "
                "after retrieval and review enrichment."
            )
        }

    # ============================================
    # 4. Ranking Agent
    # ============================================

    ranking_start = time.perf_counter()

    ranking_output = (
        ranking_agent.rank_products(
            query=structured_query,
            products=enriched_products
        )
    )

    ranking_latency = (
        time.perf_counter()
        - ranking_start
    )

    if verbose:
        print("=" * 100)
        print("4. RANKING AGENT")
        print("=" * 100)

        for product in ranking_output.products:
            print(
                f"{product.rank}. "
                f"{product.title} "
                f"| Final score: "
                f"{product.final_score:.4f}"
            )

        print()

    # ============================================
    # 5. Verifier Agent
    # ============================================

    verifier_start = time.perf_counter()

    verification_output = (
        verifier_agent.verify(
            query=structured_query,
            ranking_output=ranking_output,
            original_products=enriched_products
        )
    )

    verifier_latency = (
        time.perf_counter()
        - verifier_start
    )

    total_latency = (
        time.perf_counter()
        - pipeline_start
    )

    if verbose:
        print("=" * 100)
        print("5. VERIFIER AGENT")
        print("=" * 100)
        print(
            "Overall status:",
            verification_output.overall_status
        )
        print(
            "Confidence:",
            verification_output.confidence
        )
        print(
            "Recommended product:",
            verification_output
            .recommended_product_title
        )
        print()
        print(verification_output.summary)
        print()

        print("=" * 100)
        print("PIPELINE COMPLETED")
        print("=" * 100)
        print(
            f"Total latency: "
            f"{total_latency:.2f} seconds"
        )

    return {
        "system": "coordinated_multi_agent",
        "user_query": user_query,
        "structured_query": structured_query,
        "retrieved_products": retrieved_products,
        "enriched_products": enriched_products,
        "ranking_output": ranking_output,
        "verification_output": verification_output,
        "latency": {
            "query_agent": round(
                query_latency,
                4
            ),
            "retrieval_agent": round(
                retrieval_latency,
                4
            ),
            "review_agent": round(
                review_latency,
                4
            ),
            "ranking_agent": round(
                ranking_latency,
                4
            ),
            "verifier_agent": round(
                verifier_latency,
                4
            ),
            "total": round(
                total_latency,
                4
            )
        }
    }


In [25]:
# =====================================================
# FINAL COORDINATED MULTI-AGENT RUN
# =====================================================

from langchain_community.callbacks.manager import (
    get_openai_callback
)

import pandas as pd


FINAL_FOLDER = (
    PROJECT_ROOT
    / "Results"
    / "Final"
)


benchmark_df = pd.read_csv(
    FINAL_FOLDER
    / "final_60_query_benchmark.csv"
)


coord_final_rows = []


for _, test_case in benchmark_df.iterrows():

    test_id = int(
        test_case["test_id"]
    )

    user_query = str(
        test_case["user_query"]
    )

    print(
        "Coordinated:",
        test_id
    )


    try:

        with get_openai_callback() as cb:

            result = run_coordinated_system(
                user_query=user_query,
                retrieval_k=10,
                max_reviews=3,
                verbose=False
            )


        verification_output = result.get(
            "verification_output"
        )


        ranking_output = result.get(
            "ranking_output"
        )


        ranked_products = (
            ranking_output.products
            if ranking_output is not None
            else []
        )


        top_5_asins = [
            product.parent_asin
            for product
            in ranked_products[:5]
            if product.parent_asin
        ]


        top_5_titles = [
            product.title
            for product
            in ranked_products[:5]
            if product.title
        ]


        recommended_product = None
        recommended_asin = None
        overall_status = None
        confidence = None


        if verification_output is not None:

            recommended_product = (
                verification_output
                .recommended_product_title
            )

            recommended_asin = (
                verification_output
                .recommended_product_asin
            )

            overall_status = (
                verification_output
                .overall_status
            )

            confidence = (
                verification_output
                .confidence
            )


        coord_final_rows.append({

            "test_id":
                test_id,

            "category":
                test_case["category"],

            "user_query":
                user_query,

            "recommended_product":
                recommended_product,

            "recommended_asin":
                recommended_asin,

            "overall_status":
                overall_status,

            "confidence":
                confidence,

            "top_5_asins":
                "|".join(
                    top_5_asins
                ),

            "top_5_titles":
                " || ".join(
                    top_5_titles
                ),

            "total_latency":
                result[
                    "latency"
                ][
                    "total"
                ],

            "prompt_tokens":
                int(
                    cb.prompt_tokens
                ),

            "completion_tokens":
                int(
                    cb.completion_tokens
                ),

            "total_tokens":
                int(
                    cb.total_tokens
                ),

            "estimated_cost_usd":
                float(
                    cb.total_cost
                ),

            "error":
                result.get(
                    "error"
                )
        })


    except Exception as error:

        coord_final_rows.append({

            "test_id":
                test_id,

            "category":
                test_case["category"],

            "user_query":
                user_query,

            "recommended_product":
                None,

            "recommended_asin":
                None,

            "overall_status":
                "failed",

            "confidence":
                "low",

            "top_5_asins":
                "",

            "top_5_titles":
                "",

            "total_latency":
                0,

            "prompt_tokens":
                0,

            "completion_tokens":
                0,

            "total_tokens":
                0,

            "estimated_cost_usd":
                0,

            "error":
                str(error)
        })


coord_final_df = pd.DataFrame(
    coord_final_rows
)


display(
    coord_final_df
)


coord_final_df.to_csv(
    FINAL_FOLDER
    / "final_coordinated_results.csv",
    index=False
)


print(
    "Final Coordinated run complete."
)

Coordinated: 1
Coordinated: 2
Coordinated: 3
Coordinated: 4
Coordinated: 5
Coordinated: 6
Coordinated: 7
Coordinated: 8
Coordinated: 9
Coordinated: 10
Coordinated: 11
Coordinated: 12
Coordinated: 13
Coordinated: 14
Coordinated: 15
Coordinated: 16
Coordinated: 17
Coordinated: 18
Coordinated: 19
Coordinated: 20
Coordinated: 21
Coordinated: 22
Coordinated: 23
Coordinated: 24
Coordinated: 25
Coordinated: 26
Coordinated: 27
Coordinated: 28
Coordinated: 29
Coordinated: 30
Coordinated: 31
Coordinated: 32
Coordinated: 33
Coordinated: 34
Coordinated: 35
Coordinated: 36
Coordinated: 37
Coordinated: 38
Coordinated: 39
Coordinated: 40
Coordinated: 41
Coordinated: 42
Coordinated: 43
Coordinated: 44
Coordinated: 45
Coordinated: 46
Coordinated: 47
Coordinated: 48
Coordinated: 49
Coordinated: 50
Coordinated: 51
Coordinated: 52
Coordinated: 53
Coordinated: 54
Coordinated: 55
Coordinated: 56
Coordinated: 57
Coordinated: 58
Coordinated: 59
Coordinated: 60


,test_id,category,user_query,recommended_product,recommended_asin,overall_status,confidence,top_5_asins,top_5_titles,total_latency,prompt_tokens,completion_tokens,total_tokens,estimated_cost_usd,error
0,1,Brand Only,Recommend a Samsung phone.,Samsung Galaxy S5 SM-G900H Factory Unlocked Ce...,B00JKSUHLU,passed_with_warnings,medium,B00JKSUHLU|B00D93LOY6|B00LMJDP1Y|B00CGIULGC|B0...,Samsung Galaxy S5 SM-G900H Factory Unlocked Ce...,14.7049,3708,798,4506,0.001035,None
1,2,Brand Only,Recommend an Apple phone.,Apple iPhone 5 - 16GB (Black) Factory Unlocked,B00CX0OZHY,passed_with_warnings,medium,B00CX0OZHY,Apple iPhone 5 - 16GB (Black) Factory Unlocked,4.8089,1238,100,1338,0.000246,None
2,3,Brand Only,Suggest a Motorola smartphone.,Heart with Stars (Muti Color) Cell Phone Charm...,B004NULVAQ,passed_with_warnings,medium,B004NULVAQ|B001VF7LWI|B002VRO83K|B006P82YC8|B0...,Heart with Stars (Muti Color) Cell Phone Charm...,10.6493,2699,599,3298,0.000764,None
3,4,Brand Only,Recommend a Nokia phone.,Nokia E5-00 Unlocked GSM Phone with Easy Email...,B003X26SLM,passed_with_warnings,medium,B003X26SLM|B0055JQA46|B00UYAE1K6|B0014DGPZG,Nokia E5-00 Unlocked GSM Phone with Easy Email...,9.5178,2329,419,2748,0.000601,None
4,5,Brand Only,Suggest a Google phone.,"Google Pixel XL 128GB - 5.5"" Android GSM 4G LT...",B01M27MVQI,passed_with_warnings,medium,B01M27MVQI|B00INKCR14|B08BXBT8MD,"Google Pixel XL 128GB - 5.5"" Android GSM 4G LT...",6.8641,2045,320,2365,0.000499,None
5,6,Brand + Feature,Recommend a Samsung phone with a good camera.,Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM...,B00D93LOY6,passed_with_warnings,medium,B00D93LOY6|B09C6N8P6Y|B00CGIULGC|B0B1QXHB7L|B0...,Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM...,7.5794,2321,430,2751,0.000606,None
6,7,Brand + Feature,Recommend a Samsung phone with long battery life.,"SAMSUNG Galaxy S21 Ultra 5G, 128GB, Phantom Bl...",B09S6VKCLX,passed_with_warnings,medium,B09S6VKCLX|B09WT8N5X7|B09X9FC88M|B00JKSUHLU|B0...,"SAMSUNG Galaxy S21 Ultra 5G, 128GB, Phantom Bl...",5.9444,1798,288,2086,0.000442,None
7,8,Brand + Feature,Recommend an Apple phone with a good camera.,SteadyMate SM1 HD Professional Handheld Camera...,B00ORZVW42,passed_with_warnings,medium,B00ORZVW42,SteadyMate SM1 HD Professional Handheld Camera...,2.9600,1268,81,1349,0.000239,None
8,9,Brand + Feature,Suggest a Motorola phone with a good camera.,Motorola DROID A855 Android Phone (Verizon Wir...,B002VRO83K,passed_with_warnings,medium,B002VRO83K|B006P82YC8|B001COKAT4|B004P551BE|B0...,Motorola DROID A855 Android Phone (Verizon Wir...,3.2013,1403,139,1542,0.000294,None
9,10,Brand + Feature,Recommend a Samsung phone with fast performance.,Samsung Galaxy S5 SM-G900H Factory Unlocked Ce...,B00JKSUHLU,passed_with_warnings,medium,B00JKSUHLU|B00D93LOY6|B00LMJDP1Y|B09C6N8P6Y|B0...,Samsung Galaxy S5 SM-G900H Factory Unlocked Ce...,3.8741,1550,139,1689,0.000316,None


Final Coordinated run complete.


In [26]:
# =====================================================
# TOKEN TRACKING - COORDINATED MULTI-AGENT
# =====================================================

from langchain_community.callbacks.manager import get_openai_callback
import pandas as pd


# Load the exact same 15 queries
test_queries_df = pd.read_csv(
    PROJECT_ROOT
    / "Results"
    / "coordinated_pipeline_test_results.csv"
)[["test_id", "user_query"]].drop_duplicates(
    subset=["test_id"]
).sort_values("test_id")


coordinated_token_rows = []

print("Queries loaded:", len(test_queries_df))


for _, row in test_queries_df.iterrows():

    test_id = int(row["test_id"])
    user_query = str(row["user_query"])

    print(
        "Running Coordinated Multi-Agent:",
        test_id
    )


    with get_openai_callback() as cb:

        result = run_coordinated_system(
            user_query=user_query,
            retrieval_k=10,
            max_reviews=3,
            verbose=False
        )


    coordinated_token_rows.append({
        "test_id": test_id,
        "system": "Coordinated Multi-Agent",
        "user_query": user_query,
        "prompt_tokens": int(cb.prompt_tokens),
        "completion_tokens": int(cb.completion_tokens),
        "total_tokens": int(cb.total_tokens),
        "estimated_cost_usd": float(cb.total_cost)
    })


coordinated_token_df = pd.DataFrame(
    coordinated_token_rows
)


display(
    coordinated_token_df
)


coordinated_token_df.to_csv(
    PROJECT_ROOT
    / "Results"
    / "coordinated_token_usage.csv",
    index=False
)


print(
    "Saved Coordinated Multi-Agent token results."
)

Queries loaded: 15
Running Coordinated Multi-Agent: 1
Running Coordinated Multi-Agent: 2
Running Coordinated Multi-Agent: 3
Running Coordinated Multi-Agent: 4
Running Coordinated Multi-Agent: 5
Running Coordinated Multi-Agent: 6
Running Coordinated Multi-Agent: 7
Running Coordinated Multi-Agent: 8
Running Coordinated Multi-Agent: 9
Running Coordinated Multi-Agent: 10
Running Coordinated Multi-Agent: 11
Running Coordinated Multi-Agent: 12
Running Coordinated Multi-Agent: 13
Running Coordinated Multi-Agent: 14
Running Coordinated Multi-Agent: 15


,test_id,system,user_query,prompt_tokens,completion_tokens,total_tokens,estimated_cost_usd
0,1,Coordinated Multi-Agent,Recommend a Samsung phone with a good camera a...,1475,147,1622,0.000309
1,2,Coordinated Multi-Agent,I want an Apple phone with excellent camera qu...,1270,79,1349,0.000238
2,3,Coordinated Multi-Agent,Suggest a Google phone with fast performance.,1540,115,1655,0.000300
3,4,Coordinated Multi-Agent,Recommend a Samsung phone under 300 with a goo...,1562,157,1719,0.000328
4,5,Coordinated Multi-Agent,Recommend a smartphone under 250 with long bat...,1525,149,1674,0.000318
5,6,Coordinated Multi-Agent,Recommend a phone with a great camera.,1572,136,1708,0.000317
6,7,Coordinated Multi-Agent,Recommend a phone with long battery life.,1540,132,1672,0.000310
7,8,Coordinated Multi-Agent,Suggest a smartphone for gaming.,1534,160,1694,0.000326
8,9,Coordinated Multi-Agent,"Recommend a Samsung phone with a good camera, ...",1570,151,1721,0.000326
9,10,Coordinated Multi-Agent,Recommend an Apple phone with a good camera an...,1713,227,1940,0.000393


Saved Coordinated Multi-Agent token results.


In [27]:
result = run_coordinated_system(
    user_query=(
        "Recommend an apple headphone with a good sound quality"
        " and long battery life."
    ),
    retrieval_k=10,
    max_reviews=3,
    verbose=True
)

1. QUERY AGENT
product_type='headset' brand='Apple' budget=None features=['good sound quality', 'long battery life']

2. RETRIEVAL AGENT
Retrieved products: 1
- vodbov Bluetooth Headset, Wireless earpiece for Business/Sport/Driver Noise Cancelling Mic Black Bluetooth Headphones Compatible with Apple Android Cell Phones,MacBook and Compatible with Alexa | Score: 0.4752

3. REVIEW AGENT
Products enriched: 1
- vodbov Bluetooth Headset, Wireless earpiece for Business/Sport/Driver Noise Cancelling Mic Black Bluetooth Headphones Compatible with Apple Android Cell Phones,MacBook and Compatible with Alexa | Sentiment: Mixed

4. RANKING AGENT
1. vodbov Bluetooth Headset, Wireless earpiece for Business/Sport/Driver Noise Cancelling Mic Black Bluetooth Headphones Compatible with Apple Android Cell Phones,MacBook and Compatible with Alexa | Final score: 0.4096

5. VERIFIER AGENT
Overall status: failed
Confidence: low
Recommended product: None

Verification finished with status 'failed' and low con

In [28]:
result["structured_query"]

ProductQuery(product_type='headset', brand='Apple', budget=None, features=['good sound quality', 'long battery life'])

In [29]:
result["ranking_output"]

RankingOutput(products=[RankedProduct(rank=1, parent_asin='B07GP25L9F', title='vodbov Bluetooth Headset, Wireless earpiece for Business/Sport/Driver Noise Cancelling Mic Black Bluetooth Headphones Compatible with Apple Android Cell Phones,MacBook and Compatible with Alexa', average_rating=3.9, retrieval_score=0.475169034091271, final_score=0.409612, reason='The vodbov Bluetooth Headset ranks first due to its decent average rating of 3.9 and positive feedback as a backup option for Bluetooth devices. However, it lacks good sound quality, which is a requested feature, as noted in the review summary.')])

In [30]:
result["verification_output"]

VerifierOutput(overall_status='failed', confidence='low', verified_products=[ProductVerification(rank=1, parent_asin='B07GP25L9F', title='vodbov Bluetooth Headset, Wireless earpiece for Business/Sport/Driver Noise Cancelling Mic Black Bluetooth Headphones Compatible with Apple Android Cell Phones,MacBook and Compatible with Alexa', status='failed', brand_match=True, product_type_match=False, feature_status='partial_match', feature_match_ratio=0.5, matched_features=['good sound quality'], missing_features=['long battery life'], budget_verifiable=False, budget_match=None, evidence_strength='weak', warnings=['The product may not match the requested product type or may be an accessory.', 'Only some of the requested features were supported by the available evidence.', 'Missing feature evidence: long battery life', 'The recommendation has limited review evidence.'], verification_reason='Brand match: True. Product type match: False. Feature status: partial_match. Feature match ratio: 0.50. Ev

In [31]:
result["latency"]

{'query_agent': 0.9587,
 'retrieval_agent': 0.3952,
 'review_agent': 0.0,
 'ranking_agent': 1.548,
 'verifier_agent': 0.002,
 'total': 2.9069}

### TESTING:

In [32]:
# =====================================================
# TEST QUERIES FOR COORDINATED PIPELINE
# =====================================================

test_queries = [

    # -----------------------------------------
    # Brand + feature queries
    # -----------------------------------------
    {
        "test_id": 1,
        "category": "Brand + Features",
        "query": (
            "Recommend a Samsung phone with a good camera "
            "and long battery life."
        )
    },

    {
        "test_id": 2,
        "category": "Brand + Features",
        "query": (
            "I want an Apple phone with excellent camera quality."
        )
    },

    {
        "test_id": 3,
        "category": "Brand + Features",
        "query": (
            "Suggest a Google phone with fast performance."
        )
    },

    # -----------------------------------------
    # Budget queries
    # -----------------------------------------
    {
        "test_id": 4,
        "category": "Budget",
        "query": (
            "Recommend a Samsung phone under 300 "
            "with a good camera."
        )
    },

    {
        "test_id": 5,
        "category": "Budget",
        "query": (
            "Recommend a smartphone under 250 "
            "with long battery life."
        )
    },

    # -----------------------------------------
    # Feature-only queries
    # -----------------------------------------
    {
        "test_id": 6,
        "category": "Feature Only",
        "query": (
            "Recommend a phone with a great camera."
        )
    },

    {
        "test_id": 7,
        "category": "Feature Only",
        "query": (
            "Recommend a phone with long battery life."
        )
    },

    {
        "test_id": 8,
        "category": "Feature Only",
        "query": (
            "Suggest a smartphone for gaming."
        )
    },

    # -----------------------------------------
    # Multi-feature queries
    # -----------------------------------------
    {
        "test_id": 9,
        "category": "Multiple Features",
        "query": (
            "Recommend a Samsung phone with a good camera, "
            "long battery life and fast performance."
        )
    },

    {
        "test_id": 10,
        "category": "Multiple Features",
        "query": (
            "Recommend an Apple phone with a good camera "
            "and large storage."
        )
    },

    # -----------------------------------------
    # Ambiguous queries
    # -----------------------------------------
    {
        "test_id": 11,
        "category": "Ambiguous",
        "query": (
            "Recommend the best Samsung phone."
        )
    },

    {
        "test_id": 12,
        "category": "Ambiguous",
        "query": (
            "I need something good for photography."
        )
    },

    # -----------------------------------------
    # Negative or difficult queries
    # -----------------------------------------
    {
        "test_id": 13,
        "category": "Negative",
        "query": (
            "Recommend an Apple phone with stylus support."
        )
    },

    {
        "test_id": 14,
        "category": "Negative",
        "query": (
            "Recommend a Samsung phone with a removable battery "
            "and an excellent camera."
        )
    },

    {
        "test_id": 15,
        "category": "No Match",
        "query": (
            "Recommend a Nokia phone with an 8K camera "
            "and 1TB storage."
        )
    }
]

In [33]:
test_case = test_queries[0]

result = run_coordinated_system(
    user_query=test_case["query"],
    retrieval_k=10,
    max_reviews=3,
    verbose=True
)

1. QUERY AGENT
product_type='smartphone' brand='Samsung' budget=None features=['good camera', 'long battery life', 'camera']

2. RETRIEVAL AGENT
Retrieved products: 8
- Samsung Galaxy S21 FE 5G Cell Phone, Factory Unlocked Android Smartphone, 128GB, 120Hz Display, Pro Grade Camera, All Day Intelligent Battery, (Olive Green) (Renewed) | Score: 0.6121
- Samsung Galaxy S22 Smartphone, Factory Unlocked Android Cell Phone, 256GB, 8K Camera & Video, Brightest Display, Long Battery Life, Fast 4nm Processor, US Version, Phantom White (Renewed) | Score: 0.606
- SAMSUNG Galaxy S21 Ultra 5G, 128GB, Phantom Black - Unlocked (Renewed Premium) | Score: 0.6049
- Samsung Galaxy S5 SM-G900H Factory Unlocked Cellphone, International Version, Black | Score: 0.5535
- Samsung Galaxy Rugby Pro 4G LTE I547 Unlocked Android Ruggedized Smart Phone | Score: 0.5399
- Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM Dual-Core Android Smartphone w/ 8 MP Camera - Black | Score: 0.5394
- Samsung Galaxy S22 Ultra 256GB

In [34]:
# =====================================================
# RUN ALL TEST QUERIES
# =====================================================

import time
import pandas as pd


test_results = []

for test_case in test_queries:

    print("=" * 100)
    print(
        f"TEST {test_case['test_id']} "
        f"| {test_case['category']}"
    )
    print("=" * 100)
    print(test_case["query"])
    print()

    try:

        start_time = time.perf_counter()

        result = run_coordinated_system(
            user_query=test_case["query"],
            retrieval_k=10,
            max_reviews=3,
            verbose=False
        )

        execution_time = (
            time.perf_counter()
            - start_time
        )

        structured_query = result.get(
    "structured_query"
)

        ranking_output = result.get(
            "ranking_output"
        )

        ranked_products = (
            ranking_output.products
            if ranking_output is not None
            else []
        )

        verification_output = result.get(
            "verification_output"
        )


# =====================================================
# BUILD VERIFIED TOP-5 LIST
# =====================================================

        verified_products = (
                verification_output.verified_products
                if verification_output is not None
                else []
            )


            # Keep only products accepted by the Verifier Agent
        accepted_verified_products = [
                product
                for product in verified_products
                if str(product.status).strip().lower()
                in {
                    "passed",
                    "passed_with_warning",
                    "passed_with_warnings"
                }
            ]


            # Preserve their verified/ranking order
        accepted_verified_products = sorted(
                accepted_verified_products,
                key=lambda product: product.rank
            )


        verified_top_5 = (
                accepted_verified_products[:5]
            )


        top_5_asins = [
                product.parent_asin
                for product in verified_top_5
                if product.parent_asin
            ]


        top_5_titles = [
                product.title
                for product in verified_top_5
                if product.title
            ]

        retrieved_products = result.get(
                        "retrieved_products",
                        []
                    )
        # If verification returned no accepted products,
# keep the verified Top-5 columns empty.

        if not verified_top_5:

                top_5_asins = []
                top_5_titles = []

        if verification_output is not None:

            recommended_title = (
                verification_output.recommended_product_title
            )

            recommended_asin = (
                verification_output.recommended_product_asin
            )

            overall_status = (
                verification_output.overall_status
            )

            confidence = (
                verification_output.confidence
            )

        else:

            recommended_title = None
            recommended_asin = None
            overall_status = "no_result"
            confidence = "low"

        test_results.append(
            {
                "test_id": test_case["test_id"],
                "category": test_case["category"],
                "user_query": test_case["query"],

                "product_type": getattr(
                    structured_query,
                    "product_type",
                    None
                ),

                "brand": getattr(
                    structured_query,
                    "brand",
                    None
                ),

                "budget": getattr(
                    structured_query,
                    "budget",
                    None
                ),

                "features": ", ".join(
                    getattr(
                        structured_query,
                        "features",
                        []
                    ) or []
                ),

                "retrieved_count": len(
                    retrieved_products
                ),

                "ranked_count": len(
                    ranked_products
                ),

                "verified_count": len(
                    accepted_verified_products
                ),


                "recommended_product": recommended_title,
                "recommended_asin": recommended_asin,

                "top_5_asins": "|".join(
                    top_5_asins
                ),

                "top_5_titles": " || ".join(
                    top_5_titles
                ),

                "overall_status": overall_status,
                "confidence": confidence,

                "query_latency": result.get(
                    "latency",
                    {}
                ).get(
                    "query_agent",
                    0
                ),

                "retrieval_latency": result.get(
                    "latency",
                    {}
                ).get(
                    "retrieval_agent",
                    0
                ),

                "review_latency": result.get(
                    "latency",
                    {}
                ).get(
                    "review_agent",
                    0
                ),

                "ranking_latency": result.get(
                    "latency",
                    {}
                ).get(
                    "ranking_agent",
                    0
                ),

                "verifier_latency": result.get(
                    "latency",
                    {}
                ).get(
                    "verifier_agent",
                    0
                ),

                "total_latency": result.get(
                    "latency",
                    {}
                ).get(
                    "total",
                    execution_time
                ),

                "error": result.get("error")
            }
        )

        print(
            "Recommended:",
            recommended_title
        )

        print(
            "Status:",
            overall_status
        )

        print(
            "Latency:",
            round(execution_time, 2),
            "seconds"
        )

    except Exception as e:

        test_results.append(
            {
                "test_id": test_case["test_id"],
                "category": test_case["category"],
                "user_query": test_case["query"],

                "product_type": None,
                "brand": None,
                "budget": None,
                "features": None,

                "retrieved_count": 0,
                "ranked_count": 0,
                "verified_count": 0,

                "recommended_product": None,
                "recommended_asin": None,

                "top_5_asins": "",
                "top_5_titles": "",

                "overall_status": "error",
                "confidence": "low",

                "query_latency": 0,
                "retrieval_latency": 0,
                "review_latency": 0,
                "ranking_latency": 0,
                "verifier_latency": 0,
                "total_latency": 0,

                "error": str(e)
            }
        )

        print(
            "Error:",
            str(e)
        )

TEST 1 | Brand + Features
Recommend a Samsung phone with a good camera and long battery life.

Recommended: Samsung Galaxy S21 FE 5G Cell Phone, Factory Unlocked Android Smartphone, 128GB, 120Hz Display, Pro Grade Camera, All Day Intelligent Battery, (Olive Green) (Renewed)
Status: passed_with_warnings
Latency: 4.21 seconds
TEST 2 | Brand + Features
I want an Apple phone with excellent camera quality.

Recommended: SteadyMate SM1 HD Professional Handheld Camera Stabilizer for Apple iPhone 6 Plus, 6, 5S, 5C, 5, 4S and 4 Smart Phones
Status: passed_with_warnings
Latency: 2.55 seconds
TEST 3 | Brand + Features
Suggest a Google phone with fast performance.

Recommended: None
Status: failed
Latency: 3.58 seconds
TEST 4 | Budget
Recommend a Samsung phone under 300 with a good camera.

Recommended: Samsung Galaxy A52 (5G) 128GB A526U 6.5" Display Quad Camera Smartphone - Black (Renewed) (AT&T Unlocked)
Status: passed_with_warnings
Latency: 7.18 seconds
TEST 5 | Budget
Recommend a smartphone u

In [35]:
test_results_df = pd.DataFrame(
    test_results
)

display(
    test_results_df[
        [
            "test_id",
            "retrieved_count",
            "ranked_count",
            "verified_count",
            "recommended_product",
            "top_5_asins",
            "overall_status",
            "error"
        ]
    ]
)

,test_id,retrieved_count,ranked_count,verified_count,recommended_product,top_5_asins,overall_status,error
0,1,8,8,8,"Samsung Galaxy S21 FE 5G Cell Phone, Factory U...",B0B1QXHB7L|B09S6VKCLX|B09WT8N5X7|B00D93LOY6|B0...,passed_with_warnings,None
1,2,1,1,1,SteadyMate SM1 HD Professional Handheld Camera...,B00ORZVW42,passed_with_warnings,None
2,3,2,2,0,None,,failed,None
3,4,10,10,9,"Samsung Galaxy A52 (5G) 128GB A526U 6.5"" Displ...",B09C6N8P6Y|B00CGIULGC|B0B1QXHB7L|B00D93LOY6|B0...,passed_with_warnings,None
4,5,10,10,2,None,B0995SX8X8|B0BBRBXPGC,passed_with_warnings,None
5,6,10,10,10,"POSH Revel S500a - 5.0"", 4G, Android 4.4 Kit K...",B00O3PAN1E|B00NEEXCTK|B00CGIULGC|B002VRO83K|B0...,passed_with_warnings,None
6,7,10,10,2,OUKITEL 8000mAh Large Battery 18W Flash Charge...,B0995SX8X8|B0BBRBXPGC,passed_with_warnings,None
7,8,10,10,2,"Black Shark 4 Unlocked Phone, 5G Gaming Phone,...",B09L4QQCN2|B0BLHCXZ8K,passed_with_warnings,None
8,9,10,10,10,Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM...,B00D93LOY6|B00JKSUHLU|B0B1QXHB7L|B09S6VKCLX|B0...,passed_with_warnings,None
9,10,3,3,3,Apple iPhone 5 - 16GB (Black) Factory Unlocked,B00CX0OZHY|B00BUYRQG6|B00ORZVW42,passed_with_warnings,None


In [36]:
validation_df = test_results_df[
    [
        "test_id",
        "retrieved_count",
        "ranked_count",
        "verified_count",
        "overall_status",
        "error"
    ]
].copy()

validation_df[
    "rejected_count"
] = (
    validation_df["ranked_count"]
    - validation_df["verified_count"]
)

display(validation_df)

,test_id,retrieved_count,ranked_count,verified_count,overall_status,error,rejected_count
0,1,8,8,8,passed_with_warnings,None,0
1,2,1,1,1,passed_with_warnings,None,0
2,3,2,2,0,failed,None,2
3,4,10,10,9,passed_with_warnings,None,1
4,5,10,10,2,passed_with_warnings,None,8
5,6,10,10,10,passed_with_warnings,None,0
6,7,10,10,2,passed_with_warnings,None,8
7,8,10,10,2,passed_with_warnings,None,8
8,9,10,10,10,passed_with_warnings,None,0
9,10,3,3,3,passed_with_warnings,None,0


In [37]:
# =====================================================
# DISPLAY FINAL SUMMARY
# =====================================================

summary_columns = [
    "test_id",
    "category",
    "user_query",
    "product_type",
    "brand",
    "budget",
    "features",

    "retrieved_count",
    "ranked_count",
    "verified_count",

    "recommended_product",
    "recommended_asin",

    "top_5_asins",
    "top_5_titles",

    "overall_status",
    "confidence",
    "total_latency",
    "error"
]

display(
    test_results_df[
        summary_columns
    ]
)

,test_id,category,user_query,product_type,brand,budget,features,retrieved_count,ranked_count,verified_count,recommended_product,recommended_asin,top_5_asins,top_5_titles,overall_status,confidence,total_latency,error
0,1,Brand + Features,Recommend a Samsung phone with a good camera a...,smartphone,Samsung,NaN,"good camera, long battery life, camera",8,8,8,"Samsung Galaxy S21 FE 5G Cell Phone, Factory U...",B0B1QXHB7L,B0B1QXHB7L|B09S6VKCLX|B09WT8N5X7|B00D93LOY6|B0...,"Samsung Galaxy S21 FE 5G Cell Phone, Factory U...",passed_with_warnings,medium,4.2139,None
1,2,Brand + Features,I want an Apple phone with excellent camera qu...,smartphone,Apple,NaN,"excellent camera quality, camera",1,1,1,SteadyMate SM1 HD Professional Handheld Camera...,B00ORZVW42,B00ORZVW42,SteadyMate SM1 HD Professional Handheld Camera...,passed_with_warnings,medium,2.5485,None
2,3,Brand + Features,Suggest a Google phone with fast performance.,smartphone,Google,NaN,fast performance,2,2,0,None,None,,,failed,low,3.5827,None
3,4,Budget,Recommend a Samsung phone under 300 with a goo...,smartphone,Samsung,300.0,"good camera, camera",10,10,9,"Samsung Galaxy A52 (5G) 128GB A526U 6.5"" Displ...",B09C6N8P6Y,B09C6N8P6Y|B00CGIULGC|B0B1QXHB7L|B00D93LOY6|B0...,"Samsung Galaxy A52 (5G) 128GB A526U 6.5"" Displ...",passed_with_warnings,medium,7.1814,None
4,5,Budget,Recommend a smartphone under 250 with long bat...,smartphone,None,250.0,long battery life,10,10,2,None,None,B0995SX8X8|B0BBRBXPGC,OUKITEL 8000mAh Large Battery 18W Flash Charge...,passed_with_warnings,medium,4.3112,None
5,6,Feature Only,Recommend a phone with a great camera.,smartphone,None,NaN,"great camera, camera",10,10,10,"POSH Revel S500a - 5.0"", 4G, Android 4.4 Kit K...",B00O3PAN1E,B00O3PAN1E|B00NEEXCTK|B00CGIULGC|B002VRO83K|B0...,"POSH Revel S500a - 5.0"", 4G, Android 4.4 Kit K...",passed_with_warnings,medium,4.0805,None
6,7,Feature Only,Recommend a phone with long battery life.,smartphone,None,NaN,long battery life,10,10,2,OUKITEL 8000mAh Large Battery 18W Flash Charge...,B0995SX8X8,B0995SX8X8|B0BBRBXPGC,OUKITEL 8000mAh Large Battery 18W Flash Charge...,passed_with_warnings,medium,3.9159,None
7,8,Feature Only,Suggest a smartphone for gaming.,smartphone,None,NaN,gaming,10,10,2,"Black Shark 4 Unlocked Phone, 5G Gaming Phone,...",B09L4QQCN2,B09L4QQCN2|B0BLHCXZ8K,"Black Shark 4 Unlocked Phone, 5G Gaming Phone,...",passed_with_warnings,medium,43.6093,None
8,9,Multiple Features,"Recommend a Samsung phone with a good camera, ...",smartphone,Samsung,NaN,"good camera, long battery life, fast performan...",10,10,10,Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM...,B00D93LOY6,B00D93LOY6|B00JKSUHLU|B0B1QXHB7L|B09S6VKCLX|B0...,Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM...,passed_with_warnings,medium,3.0113,None
9,10,Multiple Features,Recommend an Apple phone with a good camera an...,smartphone,Apple,NaN,"good camera, large storage, camera",3,3,3,Apple iPhone 5 - 16GB (Black) Factory Unlocked,B00CX0OZHY,B00CX0OZHY|B00BUYRQG6|B00ORZVW42,Apple iPhone 5 - 16GB (Black) Factory Unlocked...,passed_with_warnings,medium,2.8133,None


In [38]:
# =====================================================
# SAVE UPDATED COORDINATED RESULTS
# =====================================================

from pathlib import Path

results_folder = (
    PROJECT_ROOT
    / "Results"
)

results_folder.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    results_folder
    / "coordinated_pipeline_test_results.csv"
)

test_results_df.to_csv(
    output_path,
    index=False
)

print(
    "Results saved to:",
    output_path
)

Results saved to: C:\Users\srush\Desktop\Multi agent coordination\Results\coordinated_pipeline_test_results.csv


In [39]:
print("File exists:", output_path.exists())

saved_df = pd.read_csv(output_path)

print("Rows saved:", len(saved_df))

display(
    saved_df[
        [
            "test_id",
            "verified_count",
            "recommended_product",
            "overall_status",
            "confidence"
        ]
    ]
)

File exists: True
Rows saved: 15


,test_id,verified_count,recommended_product,overall_status,confidence
0,1,8,"Samsung Galaxy S21 FE 5G Cell Phone, Factory U...",passed_with_warnings,medium
1,2,1,SteadyMate SM1 HD Professional Handheld Camera...,passed_with_warnings,medium
2,3,0,NaN,failed,low
3,4,9,"Samsung Galaxy A52 (5G) 128GB A526U 6.5"" Displ...",passed_with_warnings,medium
4,5,2,NaN,passed_with_warnings,medium
5,6,10,"POSH Revel S500a - 5.0"", 4G, Android 4.4 Kit K...",passed_with_warnings,medium
6,7,2,OUKITEL 8000mAh Large Battery 18W Flash Charge...,passed_with_warnings,medium
7,8,2,"Black Shark 4 Unlocked Phone, 5G Gaming Phone,...",passed_with_warnings,medium
8,9,10,Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM...,passed_with_warnings,medium
9,10,3,Apple iPhone 5 - 16GB (Black) Factory Unlocked,passed_with_warnings,medium


In [40]:
# =====================================================
# CONFIRM SAVED FILE
# =====================================================

saved_coordinated_df = pd.read_csv(
    output_path
)

required_columns = [
    "verified_count",
    "top_5_asins",
    "top_5_titles"
]

print(
    saved_coordinated_df[
        required_columns
    ].head()
)

print(
    "Saved rows:",
    len(saved_coordinated_df)
)

   verified_count                                        top_5_asins  \
0               8  B0B1QXHB7L|B09S6VKCLX|B09WT8N5X7|B00D93LOY6|B0...   
1               1                                         B00ORZVW42   
2               0                                                NaN   
3               9  B09C6N8P6Y|B00CGIULGC|B0B1QXHB7L|B00D93LOY6|B0...   
4               2                              B0995SX8X8|B0BBRBXPGC   

                                        top_5_titles  
0  Samsung Galaxy S21 FE 5G Cell Phone, Factory U...  
1  SteadyMate SM1 HD Professional Handheld Camera...  
2                                                NaN  
3  Samsung Galaxy A52 (5G) 128GB A526U 6.5" Displ...  
4  OUKITEL 8000mAh Large Battery 18W Flash Charge...  
Saved rows: 15


In [41]:
# =====================================================
# DIAGNOSE TEST 8: GAMING QUERY
# =====================================================

gaming_result = run_coordinated_system(
    user_query="Suggest a smartphone for gaming.",
    retrieval_k=10,
    max_reviews=3,
    verbose=False
)

structured_query = gaming_result[
    "structured_query"
]

print("=" * 80)
print("STRUCTURED QUERY")
print("=" * 80)

if hasattr(
    structured_query,
    "model_dump"
):
    print(
        structured_query.model_dump()
    )
else:
    print(
        structured_query
    )


print()
print("=" * 80)
print("RETRIEVED PRODUCTS")
print("=" * 80)

for rank, (
    document,
    score
) in enumerate(
    gaming_result.get(
        "retrieved_products",
        []
    ),
    start=1
):

    print(
        rank,
        "|",
        document.metadata.get(
            "parent_asin"
        ),
        "|",
        document.metadata.get(
            "title"
        ),
        "| score:",
        round(
            float(score),
            4
        )
    )


print()
print("=" * 80)
print("RANKED PRODUCTS")
print("=" * 80)

ranking_output = gaming_result.get(
    "ranking_output"
)

if ranking_output is not None:

    for product in (
        ranking_output.products
    ):

        print(
            product.rank,
            "|",
            product.parent_asin,
            "|",
            product.title,
            "| final score:",
            round(
                product.final_score,
                4
            )
        )


print()
print("=" * 80)
print("VERIFIED PRODUCTS")
print("=" * 80)

verification_output = gaming_result.get(
    "verification_output"
)

if verification_output is not None:

    for product in (
        verification_output
        .verified_products
    ):

        print(
            product.rank,
            "|",
            product.parent_asin,
            "|",
            product.title,
            "| status:",
            product.status
        )

STRUCTURED QUERY
{'product_type': 'smartphone', 'brand': None, 'budget': None, 'features': ['gaming']}

RETRIEVED PRODUCTS
1 | B09L4QQCN2 | Black Shark 4 Unlocked Phone, 5G Gaming Phone, Fast Charging 120W Cell Phone 12+256GB, 144Hz Snapdragon 870 Android Phone, 6.67" 48MP 4500mAh NFC Mobile Phone Global Version -Frost Grey | score: 0.6329
2 | B0BLHCXZ8K | SANSHREUNI Dual Sim Unlocked Cell Phones,4G+128GB C21 Ultra Smartphone, 6.8 inch Waterdrop Screen Android Phone, 5000mAh Battery, 24+50 MP Unlocked Cell Phone, Fingerprint Lock & Face ID Mobile Phone | score: 0.5417
3 | B00NEEXCTK | Huawei Ascend G7 16GB GSM Unlocked Android Smartphone w/ 13MP Camera, 5.5" IPS LCD Display, Quad-Core CPU - White/Silver | score: 0.4675
4 | B00NO80S3E | HTC Deluxe - 4G LTE GSM Factory Unlocked, 5" Android Smartphone with Beats Audio - Black | score: 0.4561
5 | B002VRO83K | Motorola DROID A855 Android Phone (Verizon Wireless) | score: 0.4517
6 | B00CGIULGC | Samsung Galaxy Rugby Pro 4G LTE I547 Unlocked 

In [42]:
# =====================================================
# CHECK WHETHER BLACK SHARK EXISTS IN FAISS
# =====================================================

exact_results = vector_db.similarity_search_with_score(
    "Black Shark 4 5G Gaming Phone",
    k=50
)

black_shark_matches = []

for document, score in exact_results:

    asin = str(
        document.metadata.get(
            "parent_asin",
            ""
        )
    ).strip()

    title = str(
        document.metadata.get(
            "title",
            ""
        )
    ).strip()

    if (
        asin == "B09L4QQCN2"
        or "black shark" in title.lower()
    ):
        black_shark_matches.append(
            {
                "parent_asin": asin,
                "title": title,
                "score": float(score)
            }
        )

print(
    "Black Shark matches found:",
    len(black_shark_matches)
)

for match in black_shark_matches:
    print(match)

Black Shark matches found: 1
{'parent_asin': 'B09L4QQCN2', 'title': 'Black Shark 4 Unlocked Phone, 5G Gaming Phone, Fast Charging 120W Cell Phone 12+256GB, 144Hz Snapdragon 870 Android Phone, 6.67" 48MP 4500mAh NFC Mobile Phone Global Version -Frost Grey', 'score': 0.5815507173538208}


In [43]:
# =====================================================
# RAW FAISS SEARCH FOR GAMING QUERY
# =====================================================

raw_gaming_results = (
    vector_db.similarity_search_with_score(
        "smartphone gaming",
        k=50
    )
)

for rank, (
    document,
    score
) in enumerate(
    raw_gaming_results,
    start=1
):

    asin = document.metadata.get(
        "parent_asin"
    )

    title = document.metadata.get(
        "title"
    )

    print(
        rank,
        "|",
        asin,
        "|",
        title,
        "| score:",
        round(
            float(score),
            4
        )
    )

1 | B09L4QQCN2 | Black Shark 4 Unlocked Phone, 5G Gaming Phone, Fast Charging 120W Cell Phone 12+256GB, 144Hz Snapdragon 870 Android Phone, 6.67" 48MP 4500mAh NFC Mobile Phone Global Version -Frost Grey | score: 1.0818
2 | B07F4BSXMQ | Gamer Heartbeat Smartphone Grip Best Video Game Player Gift PopSockets Grip and Stand for Phones and Tablets | score: 1.1072
3 | B018C96B7Q | Airsspu Virtual Reality Headset 3D VR Glasses Google Cardboard for iPhone Samsung Note LG HTC Moto 4~6 inch Smartphones for 3D Video/Movies/Games | score: 1.1456
4 | B07L2W1ZSR | Gameboy iPhone Case Handheld Game Console Phone Case with 36 Small Games Color Screen Retro 3D Gameboy Design for iPhone Xs Max, White | score: 1.1487
5 | B01768GWC8 | SainSonic ZH-01 Virtual Reality Headset 3D VR Glasses for 3.5~5.6 inch Smartphone | score: 1.1643
6 | B08N47M2DV | Autbye Gameboy Phone Cases, Retro 3D Gameboy Case for iPhone with 36 Small Games, Color Display Shockproof Video Game Phone Case, Phone Protective Case(for iPho

In [44]:
# =====================================================
# DIAGNOSTIC QUERIES
# =====================================================

diagnostic_queries = [
    {
        "test_id": 6,
        "query": "Recommend a phone with a great camera.",
        "expected_terms": [
            "pixel",
            "s21 ultra",
            "camera"
        ]
    },
    {
        "test_id": 7,
        "query": "Recommend a phone with long battery life.",
        "expected_terms": [
            "8000mah",
            "6000mah",
            "max",
            "battery"
        ]
    },
    {
        "test_id": 8,
        "query": "Suggest a smartphone for gaming.",
        "expected_terms": [
            "black shark",
            "gaming",
            "rog",
            "redmagic"
        ]
    },
    {
        "test_id": 9,
        "query": (
            "Recommend a Samsung phone with a good camera, "
            "long battery life and fast performance."
        ),
        "expected_terms": [
            "s22",
            "s21",
            "s10"
        ]
    },
    {
        "test_id": 12,
        "query": "I need something good for photography.",
        "expected_terms": [
            "pixel",
            "s21 ultra",
            "camera phone"
        ]
    },
    {
        "test_id": 14,
        "query": (
            "Recommend a Samsung phone with a removable battery "
            "and an excellent camera."
        ),
        "expected_terms": [
            "galaxy s5",
            "removable battery"
        ]
    }
]

In [45]:
# =====================================================
# RAW FAISS VS RETRIEVAL AGENT
# =====================================================

def diagnose_retrieval_query(
    test_id: int,
    user_query: str,
    expected_terms: list[str],
    raw_k: int = 30,
    retrieval_k: int = 10
) -> dict:

    structured_query = query_agent.parse(
        user_query
    )

    print()
    print("=" * 100)
    print(
        f"TEST {test_id}: {user_query}"
    )
    print("=" * 100)

    print(
        "Structured query:",
        structured_query.model_dump()
        if hasattr(
            structured_query,
            "model_dump"
        )
        else structured_query
    )

    # Raw FAISS search
    raw_search_text = " ".join(
        [
            str(
                structured_query.product_type
                or ""
            ),
            str(
                structured_query.brand
                or ""
            ),
            " ".join(
                structured_query.features
                or []
            )
        ]
    ).strip()

    raw_results = (
        vector_db.similarity_search_with_score(
            raw_search_text,
            k=raw_k
        )
    )

    # Final Retrieval Agent output
    agent_results = (
        retrieval_agent.retrieve(
            query=structured_query,
            k=retrieval_k
        )
    )

    expected_terms_lower = [
        term.lower()
        for term in expected_terms
    ]

    def matches_expected(
        title: str
    ) -> bool:

        title_lower = str(
            title
        ).lower()

        return any(
            term in title_lower
            for term in expected_terms_lower
        )

    raw_expected = []

    for rank, (
        document,
        score
    ) in enumerate(
        raw_results,
        start=1
    ):

        title = document.metadata.get(
            "title",
            ""
        )

        if matches_expected(title):

            raw_expected.append(
                {
                    "rank": rank,
                    "asin": (
                        document.metadata.get(
                            "parent_asin"
                        )
                    ),
                    "title": title,
                    "score": float(score)
                }
            )

    agent_expected = []

    for rank, (
        document,
        score
    ) in enumerate(
        agent_results,
        start=1
    ):

        title = document.metadata.get(
            "title",
            ""
        )

        if matches_expected(title):

            agent_expected.append(
                {
                    "rank": rank,
                    "asin": (
                        document.metadata.get(
                            "parent_asin"
                        )
                    ),
                    "title": title,
                    "score": float(score)
                }
            )

    print()
    print("Expected matches in raw FAISS:")
    for product in raw_expected:
        print(product)

    print()
    print("Expected matches after Retrieval Agent:")
    for product in agent_expected:
        print(product)

    if raw_expected and not agent_expected:
        diagnosis = (
            "Potential retrieval filtering loss"
        )

    elif raw_expected and agent_expected:
        diagnosis = (
            "Expected product preserved"
        )

    elif not raw_expected:
        diagnosis = (
            "Expected product not found in raw candidate pool"
        )

    else:
        diagnosis = (
            "No clear diagnosis"
        )

    print()
    print("Diagnosis:", diagnosis)

    return {
        "test_id": test_id,
        "user_query": user_query,
        "structured_features": "|".join(
            structured_query.features
            or []
        ),
        "raw_expected_count": len(
            raw_expected
        ),
        "agent_expected_count": len(
            agent_expected
        ),
        "diagnosis": diagnosis
    }

In [46]:
diagnostic_results = []

for test_case in diagnostic_queries:

    result = diagnose_retrieval_query(
        test_id=test_case["test_id"],
        user_query=test_case["query"],
        expected_terms=(
            test_case["expected_terms"]
        ),
        raw_k=30,
        retrieval_k=10
    )

    diagnostic_results.append(
        result
    )

diagnostic_df = pd.DataFrame(
    diagnostic_results
)

display(
    diagnostic_df
)


TEST 6: Recommend a phone with a great camera.
Structured query: {'product_type': 'smartphone', 'brand': None, 'budget': None, 'features': ['great camera', 'camera']}

Expected matches in raw FAISS:
{'rank': 2, 'asin': 'B07979NV1M', 'title': 'ShutterGrip Secure Camera Handle Holder with Removable Bluetooth Remote Clicker, FaceTime, Zoom Stand, Tripod Compatible Bluetooth Camera Remote for iPhone, Android', 'score': 1.0410563945770264}
{'rank': 5, 'asin': 'B06XQ41H62', 'title': 'Premium HD Bluetooth Selfie Remote Control Camera Shutter for iPhone, Samsung Galaxy, Android, iPad, iPod, Tablets - Amazing Selfie Clicker for Photos, Videos, 30ft Range (Aqua)', 'score': 1.0664548873901367}
{'rank': 6, 'asin': 'B07QYTKZMT', 'title': 'UKCOCO Phone Camera Lens Compatible with iPhone, Samsung and Other Smartphones-Universal Portable Lens Kit Super Wide Angle Lens Macro Lens and Fisheye Lens Clip On 3 in 1 Mobile Phone Lens(Red)', 'score': 1.0671354532241821}
{'rank': 8, 'asin': 'B00USQ4UX4', 'ti

,test_id,user_query,structured_features,raw_expected_count,agent_expected_count,diagnosis
0,6,Recommend a phone with a great camera.,great camera|camera,16,3,Expected product preserved
1,7,Recommend a phone with long battery life.,long battery life,22,5,Expected product preserved
2,8,Suggest a smartphone for gaming.,gaming,3,1,Expected product preserved
3,9,"Recommend a Samsung phone with a good camera, ...",good camera|long battery life|fast performance...,7,5,Expected product preserved
4,12,I need something good for photography.,photography,1,0,Potential retrieval filtering loss
5,14,Recommend a Samsung phone with a removable bat...,removable battery|excellent camera|camera,2,1,Expected product preserved


In [47]:
# =====================================================
# DIAGNOSE TEST 14
# =====================================================

test_query = (
    "Recommend a Samsung phone with a removable "
    "battery and an excellent camera."
)

test_query_structured = query_agent.parse(
    test_query
)

print(
    "Structured query:",
    test_query_structured.model_dump()
)

search_query = build_search_query(
    test_query_structured
)

print(
    "\nSearch query:",
    search_query
)

# Get a large raw FAISS candidate pool
raw_results = (
    vector_db.similarity_search_with_score(
        search_query,
        k=50
    )
)

target_asin = "B00JKSUHLU"


# =====================================================
# 1. FIND TARGET IN RAW FAISS
# =====================================================

print("\nRAW FAISS")

for rank, (doc, score) in enumerate(
    raw_results,
    start=1
):

    asin = str(
        doc.metadata.get(
            "parent_asin",
            ""
        )
    )

    if asin == target_asin:

        print(
            "Found target at raw rank:",
            rank
        )

        print(
            "Title:",
            doc.metadata.get(
                "title"
            )
        )

        print(
            "Rating:",
            doc.metadata.get(
                "average_rating"
            )
        )

        print(
            "Price:",
            doc.metadata.get(
                "price"
            )
        )

        print(
            "FAISS distance:",
            score
        )


# =====================================================
# 2. APPLY PRODUCT FILTER
# =====================================================

filtered_results = filter_products(
    raw_results,
    test_query_structured,
    min_rating=0.0
)

filtered_asins = [
    str(
        doc.metadata.get(
            "parent_asin",
            ""
        )
    )
    for doc, _ in filtered_results
]

print()
print(
    "Target survived filter:",
    target_asin in filtered_asins
)

print(
    "Products after filtering:",
    len(filtered_results)
)


# =====================================================
# 3. RUN FINAL RETRIEVAL AGENT
# =====================================================

final_results = retrieval_agent.retrieve(
    query=test_query_structured,
    k=10
)

print()
print("FINAL RETRIEVAL")

for rank, (
    doc,
    score
) in enumerate(
    final_results,
    start=1
):

    print(
        rank,
        "|",
        doc.metadata.get(
            "parent_asin"
        ),
        "|",
        doc.metadata.get(
            "title"
        ),
        "|",
        round(
            float(score),
            4
        )
    )


final_asins = [
    str(
        doc.metadata.get(
            "parent_asin",
            ""
        )
    )
    for doc, _ in final_results
]

print()
print(
    "Target in final Top-10:",
    target_asin in final_asins
)

Structured query: {'product_type': 'smartphone', 'brand': 'Samsung', 'budget': None, 'features': ['removable battery', 'excellent camera', 'camera']}

Search query: phone smartphone mobile cell phone cellphone android phone samsung removable battery camera

RAW FAISS
Found target at raw rank: 25
Title: Samsung Galaxy S5 SM-G900H Factory Unlocked Cellphone, International Version, Black
Rating: 4.3
Price: 579.99
FAISS distance: 1.0417097

Target survived filter: True
Products after filtering: 7

FINAL RETRIEVAL
1 | B00JKSUHLU | Samsung Galaxy S5 SM-G900H Factory Unlocked Cellphone, International Version, Black | 0.5558
2 | B00CGIULGC | Samsung Galaxy Rugby Pro 4G LTE I547 Unlocked Android Ruggedized Smart Phone | 0.5506
3 | B00D93LOY6 | Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM Dual-Core Android Smartphone w/ 8 MP Camera - Black | 0.5443
4 | B09X9FC88M | Samsung Galaxy S22 Ultra 256GB Unlocked ATT TMobile Verizon 100% batt! 7548400 | 0.5234
5 | B09WT8N5X7 | Samsung Galaxy S22 Smartp

In [48]:
# =====================================================
# RETRIEVAL REGRESSION TEST
# =====================================================

test_queries = [
    "Suggest a smartphone for gaming.",

    "Recommend a phone with a great camera.",

    "Recommend a phone with long battery life.",

    (
        "Recommend a Samsung phone with a good camera, "
        "long battery life and fast performance."
    ),

    (
        "Recommend a Samsung phone with a removable battery "
        "and an excellent camera."
    )
]


for text in test_queries:

    query = query_agent.parse(
        text
    )

    print("\n" + "=" * 100)
    print("QUERY:", text)

    print(
        "STRUCTURED:",
        query.model_dump()
    )

    results = retrieval_agent.retrieve(
        query=query,
        k=10
    )

    print("\nRETRIEVED PRODUCTS:")

    for rank, (
        doc,
        score
    ) in enumerate(
        results,
        start=1
    ):

        print(
            rank,
            "|",
            doc.metadata.get(
                "parent_asin"
            ),
            "|",
            doc.metadata.get(
                "title"
            ),
            "|",
            round(
                float(score),
                4
            )
        )


QUERY: Suggest a smartphone for gaming.
STRUCTURED: {'product_type': 'smartphone', 'brand': None, 'budget': None, 'features': ['gaming']}

RETRIEVED PRODUCTS:
1 | B09L4QQCN2 | Black Shark 4 Unlocked Phone, 5G Gaming Phone, Fast Charging 120W Cell Phone 12+256GB, 144Hz Snapdragon 870 Android Phone, 6.67" 48MP 4500mAh NFC Mobile Phone Global Version -Frost Grey | 0.6329
2 | B0BLHCXZ8K | SANSHREUNI Dual Sim Unlocked Cell Phones,4G+128GB C21 Ultra Smartphone, 6.8 inch Waterdrop Screen Android Phone, 5000mAh Battery, 24+50 MP Unlocked Cell Phone, Fingerprint Lock & Face ID Mobile Phone | 0.5417
3 | B00NEEXCTK | Huawei Ascend G7 16GB GSM Unlocked Android Smartphone w/ 13MP Camera, 5.5" IPS LCD Display, Quad-Core CPU - White/Silver | 0.4675
4 | B00NO80S3E | HTC Deluxe - 4G LTE GSM Factory Unlocked, 5" Android Smartphone with Beats Audio - Black | 0.4561
5 | B002VRO83K | Motorola DROID A855 Android Phone (Verizon Wireless) | 0.4517
6 | B00CGIULGC | Samsung Galaxy Rugby Pro 4G LTE I547 Unlocke

In [49]:
# =====================================================
# TEST UPDATED VERIFIER
# =====================================================

verifier_test_queries = [
    "Suggest a smartphone for gaming.",

    (
        "Recommend a Samsung phone with a good camera, "
        "long battery life and fast performance."
    ),

    (
        "Recommend a Samsung phone with a removable battery "
        "and an excellent camera."
    ),

    (
        "Recommend a Nokia phone with an 8K camera "
        "and 1TB storage."
    )
]


for text in verifier_test_queries:

    print("\n" + "=" * 100)
    print("QUERY:", text)
    print("=" * 100)

    result = run_coordinated_system(
        user_query=text,
        retrieval_k=10,
        max_reviews=3,
        verbose=False
    )

    verification_output = result.get(
        "verification_output"
    )

    if verification_output is None:

        print("NO VERIFICATION OUTPUT")
        continue

    print(
        "Overall status:",
        verification_output.overall_status
    )

    print(
        "Recommended product:",
        verification_output.recommended_product_title
    )

    print(
        "Recommended ASIN:",
        verification_output.recommended_product_asin
    )

    print(
        "Confidence:",
        verification_output.confidence
    )

    print("\nVERIFIED PRODUCTS:")

    for product in (
        verification_output.verified_products
    ):

        print(
            product.rank,
            "|",
            product.parent_asin,
            "|",
            product.status,
            "| feature:",
            product.feature_status,
            "| ratio:",
            product.feature_match_ratio,
            "| missing:",
            product.missing_features
        )


QUERY: Suggest a smartphone for gaming.
Overall status: passed_with_warnings
Recommended product: Black Shark 4 Unlocked Phone, 5G Gaming Phone, Fast Charging 120W Cell Phone 12+256GB, 144Hz Snapdragon 870 Android Phone, 6.67" 48MP 4500mAh NFC Mobile Phone Global Version -Frost Grey
Recommended ASIN: B09L4QQCN2
Confidence: medium

VERIFIED PRODUCTS:
1 | B09L4QQCN2 | passed_with_warning | feature: full_match | ratio: 1.0 | missing: []
2 | B0BLHCXZ8K | passed_with_warning | feature: full_match | ratio: 1.0 | missing: []
3 | B00O3PAN1E | failed | feature: no_match | ratio: 0.0 | missing: ['gaming']
4 | B00NEEXCTK | failed | feature: no_match | ratio: 0.0 | missing: ['gaming']
5 | B00NO80S3E | failed | feature: no_match | ratio: 0.0 | missing: ['gaming']
6 | B00CGIULGC | failed | feature: no_match | ratio: 0.0 | missing: ['gaming']
7 | B002VRO83K | failed | feature: no_match | ratio: 0.0 | missing: ['gaming']
8 | B07RB4ZH83 | failed | feature: no_match | ratio: 0.0 | missing: ['gaming']
9

In [50]:
result = run_coordinated_system(
    user_query=(
        "Recommend a Nokia phone with an 8K camera "
        "and 1TB storage."
    ),
    retrieval_k=10,
    max_reviews=3,
    verbose=False
)

print(result)

verification_output = result.get(
    "verification_output"
)

print("\nVerification output:")
print(verification_output)

if verification_output is not None:

    print(
        "\nOverall status:",
        verification_output.overall_status
    )

    print(
        "Recommended product:",
        verification_output.recommended_product_title
    )

    print(
        "Recommended ASIN:",
        verification_output.recommended_product_asin
    )

    for product in (
        verification_output.verified_products
    ):
        print(
            product.rank,
            "|",
            product.parent_asin,
            "|",
            product.status,
            "|",
            product.feature_status,
            "|",
            product.missing_features
        )

{'system': 'coordinated_multi_agent', 'user_query': 'Recommend a Nokia phone with an 8K camera and 1TB storage.', 'structured_query': ProductQuery(product_type='smartphone', brand='Nokia', budget=None, features=['8k camera', '1tb storage']), 'retrieved_products': [(Document(id='018822ba-1feb-4eb3-9f27-02897d9ddc67', metadata={'parent_asin': 'B003X26SLM', 'title': 'Nokia E5-00 Unlocked GSM Phone with Easy Email Setup, IM, QWERTY, 5 MP Camera, Ovi Store with Apps, and Free Ovi Maps Navigation (White)', 'price': nan, 'average_rating': 4.4, 'rating_number': 139, 'categories': 'Cell Phones & Accessories > Wireless Trade-In Buy Box Widget > Unlocked Wireless Trade-In', 'similarity': 0.4940051158919844, 'retrieval_score': 0.5463256302487554, 'feature_match': 0.5, 'brand': 'nokia', 'product_type': 'phone', 'requested_features': ['camera', '1tb storage'], 'budget': None}, page_content='Title: Nokia E5-00 Unlocked GSM Phone with Easy Email Setup, IM, QWERTY, 5 MP Camera, Ovi Store with Apps, and

In [51]:
# =====================================================
# COORDINATED SCALABILITY TEST
# =====================================================

from pathlib import Path
import pandas as pd

from langchain_community.callbacks.manager import (
    get_openai_callback
)


PROJECT_ROOT = Path("..").resolve()

FINAL_FOLDER = (
    PROJECT_ROOT
    / "Results"
    / "Final"
)


benchmark = pd.read_csv(
    FINAL_FOLDER
    / "final_60_query_benchmark.csv"
)


SCALABILITY_QUERY_IDS = [
    6,
    14,
    21,
    29,
    34,
    37,
    48,
    54
]


scalability_queries = (
    benchmark[
        benchmark[
            "test_id"
        ].isin(
            SCALABILITY_QUERY_IDS
        )
    ]
    .copy()
)


print(
    "Scalability queries:",
    len(scalability_queries)
)


scalability_rows = []


for k in [5, 10, 20, 30]:

    print(
        "\nRunning retrieval_k =",
        k
    )

    for _, row in scalability_queries.iterrows():

        print(
            "Test:",
            row["test_id"]
        )

        with get_openai_callback() as cb:

            result = run_coordinated_system(
                user_query=row[
                    "user_query"
                ],
                retrieval_k=k,
                max_reviews=3,
                verbose=False
            )


        scalability_rows.append({

            "system":
                "Coordinated Multi-Agent",

            "test_id":
                int(
                    row[
                        "test_id"
                    ]
                ),

            "retrieval_k":
                k,

            "latency":
                result[
                    "latency"
                ][
                    "total"
                ],

            "tokens":
                int(
                    cb.total_tokens
                )
        })


coordinated_scalability = pd.DataFrame(
    scalability_rows
)


display(
    coordinated_scalability
)


coordinated_scalability.to_csv(
    FINAL_FOLDER
    / "coordinated_scalability.csv",
    index=False
)


print(
    "Saved coordinated scalability results."
)

Scalability queries: 8

Running retrieval_k = 5
Test: 6
Test: 14
Test: 21
Test: 29
Test: 34
Test: 37
Test: 48
Test: 54

Running retrieval_k = 10
Test: 6
Test: 14
Test: 21
Test: 29
Test: 34
Test: 37
Test: 48
Test: 54

Running retrieval_k = 20
Test: 6
Test: 14
Test: 21
Test: 29
Test: 34
Test: 37
Test: 48
Test: 54

Running retrieval_k = 30
Test: 6
Test: 14
Test: 21
Test: 29
Test: 34
Test: 37
Test: 48
Test: 54


,system,test_id,retrieval_k,latency,tokens
0,Coordinated Multi-Agent,6,5,5.2852,1728
1,Coordinated Multi-Agent,14,5,3.5117,1705
2,Coordinated Multi-Agent,21,5,4.0063,1720
3,Coordinated Multi-Agent,29,5,3.5405,1652
4,Coordinated Multi-Agent,34,5,4.1996,1647
5,Coordinated Multi-Agent,37,5,4.2245,1717
6,Coordinated Multi-Agent,48,5,3.6629,1339
7,Coordinated Multi-Agent,54,5,4.6217,1697
8,Coordinated Multi-Agent,6,10,4.0924,1737
9,Coordinated Multi-Agent,14,10,3.7795,1703


Saved coordinated scalability results.


In [52]:
coordinated_scalability.groupby(
    "retrieval_k"
)[
    [
        "latency",
        "tokens"
    ]
].mean()

,latency,tokens
retrieval_k,,
5,4.131550,1650.625
10,4.080650,1638.500
20,6.258025,1969.500
30,6.496462,2015.375
